# Fine-Tune Qwen2.5-VL-7B for Bank Check Extraction

LoRA fine-tuning on 100 labeled check images using Unsloth + QLoRA on T4 GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have `training_data.zip` ready (created by `prepare_colab_upload.py`)
3. Run cells in order — do not skip

In [ ]:
# Cell 1 — Environment Setup
# Unsloth handles torch, transformers, bitsandbytes, peft, trl internally.
# Do NOT pip install torch or bitsandbytes separately.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "No GPU — change runtime to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import unsloth
print(f"Unsloth version: {unsloth.__version__}")
print("\nCell 1 PASSED")

In [ ]:
# Cell 2 — Upload and Extract Training Data
from google.colab import files
import zipfile, os

print("Upload training_data.zip...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Handle both flat and nested zip structures
if os.path.exists('images') and not os.path.exists('training/images'):
    os.makedirs('training', exist_ok=True)
    os.rename('images', 'training/images')
    if os.path.exists('holdout'):
        os.rename('holdout', 'training/holdout')

train_imgs = [f for f in os.listdir('training/images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
holdout_imgs = [f for f in os.listdir('training/holdout') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Training images: {len(train_imgs)}")
print(f"Holdout images:  {len(holdout_imgs)}")
assert len(train_imgs) >= 50, f"Need at least 50 training images, found {len(train_imgs)}"
print("\nCell 2 PASSED")

In [ ]:
# Cell 3 — HuggingFace Login (optional for Qwen2.5-VL but helps with rate limits)
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = input("Paste your HuggingFace token (or press Enter to skip): ").strip()

if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")
else:
    print("Skipped HF login — Qwen2.5-VL is not gated so this is fine")
print("\nCell 3 PASSED")

In [ ]:
# Cell 4 — Build Training JSONL
import json, base64
from pathlib import Path

SYSTEM_PROMPT = """You are a bank check data extraction model. Given an image of a check, extract the following fields and return ONLY a valid JSON object with no markdown, no code fences, no explanation. Use null (not empty string) for any field you cannot find or read.

Fields to extract:
- payorInstitution: The name of the bank printed on the check (e.g. "BANK OF AMERICA")
- payor: The account holder who wrote the check (printed name/address in top-left)
- payee: The person or entity on the "Pay to the order of" line
- amount: Dollar amount as a string with two decimal places (e.g. "2675.00")
- account: Account number from the MICR line at the bottom of the check
- serial: Check serial/number (typically top-right or MICR line)
- checkDate: Date written on the check in YYYY-MM-DD format
- fractionalNumber: The fractional routing number near the check number, format: DD-DDD/DDDD (e.g. "87-176/843"). This is NOT the 9-digit routing number.
- calculatedRoutingNumber: Leave this as null — it will be computed in post-processing."""

USER_PROMPT = "Extract all fields from this bank check image and return a JSON object."

def build_jsonl(image_dir, output_path):
    samples = []
    img_dir = Path(image_dir)

    for img_path in sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))):
        label_path = img_path.with_suffix('.json')
        if not label_path.exists():
            print(f"  WARNING: no label for {img_path.name}, skipping")
            continue

        with open(label_path) as f:
            label = json.load(f)

        # Force calculatedRoutingNumber to null — computed in post-processing
        label['calculatedRoutingNumber'] = None

        with open(img_path, 'rb') as f:
            img_b64 = base64.b64encode(f.read()).decode('utf-8')

        samples.append({
            'image_path': str(img_path),
            'image_b64': img_b64,
            'ground_truth': json.dumps(label, indent=2)
        })

    with open(output_path, 'w') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')

    print(f"  {len(samples)} samples -> {output_path}")
    return samples

print("Building datasets...")
train_samples = build_jsonl('training/images', 'train.jsonl')
holdout_samples = build_jsonl('training/holdout', 'holdout.jsonl')
print(f"\nTotal: {len(train_samples)} train + {len(holdout_samples)} holdout")
print("\nCell 4 PASSED")

In [ ]:
# Cell 5 — Load Model with Unsloth
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen2.5-VL-7B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("Model loaded successfully")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("\nCell 5 PASSED")

In [ ]:
# Cell 6 — Apply LoRA Adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=False,
)

model.print_trainable_parameters()
print("\nCell 6 PASSED")

In [ ]:
# Cell 7 — Dataset and Sanity Check
# Unsloth handles vision tokenization internally via SFTTrainer.
# We just need to provide messages with image file paths.
from torch.utils.data import Dataset
from PIL import Image
import io, base64, json, os

class CheckDataset(Dataset):
    def __init__(self, jsonl_path):
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                self.samples.append(json.loads(line.strip()))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Save image to a temp file if not already on disk
        # (Qwen2.5-VL processor needs file paths, not PIL objects)
        img_path = sample['image_path']
        if not os.path.exists(img_path):
            img_bytes = base64.b64decode(sample['image_b64'])
            os.makedirs('/content/temp_images', exist_ok=True)
            img_path = f'/content/temp_images/img_{idx}.jpg'
            with open(img_path, 'wb') as f:
                f.write(img_bytes)

        return {
            "messages": [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": SYSTEM_PROMPT}]
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img_path},
                        {"type": "text", "text": USER_PROMPT}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": sample['ground_truth']}]
                }
            ]
        }


train_dataset = CheckDataset('train.jsonl')
print(f"Training samples: {len(train_dataset)}")

# Sanity check — verify one sample loads correctly
sample = train_dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"Message roles: {[m['role'] for m in sample['messages']]}")
print(f"Image path exists: {os.path.exists(sample['messages'][1]['content'][0]['image'])}")
print(f"Ground truth length: {len(sample['messages'][2]['content'][0]['text'])} chars")
print("\nCell 7 PASSED")

In [ ]:
# Cell 8 — Baseline Evaluation (Before Training)
import re

FastVisionModel.for_inference(model)

def run_inference(image, model, tokenizer):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": USER_PROMPT}
            ]
        }
    ]

    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        text=[text_input],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.1
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    response = re.sub(r'^```(?:json)?\s*', '', response)
    response = re.sub(r'\s*```$', '', response)
    response = response.strip()

    try:
        return json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return {"error": "JSON parse failed", "raw": response[:300]}


SCORE_FIELDS = ['payorInstitution', 'payor', 'payee', 'amount', 'account',
                'serial', 'checkDate', 'fractionalNumber']

def score_extraction(predicted, ground_truth):
    scores = {}
    for field in SCORE_FIELDS:
        gt_val = ground_truth.get(field)
        pred_val = predicted.get(field)

        if gt_val is None and pred_val is None:
            scores[field] = 'correct_null'
        elif gt_val is None:
            scores[field] = 'false_positive'
        elif pred_val is None:
            scores[field] = 'missing'
        elif str(gt_val).strip().lower() == str(pred_val).strip().lower():
            scores[field] = 'exact_match'
        else:
            scores[field] = f'mismatch (gt={gt_val!r}, pred={pred_val!r})'

    exact = sum(1 for v in scores.values() if v in ('exact_match', 'correct_null'))
    return {'field_scores': scores, 'completeness': exact / len(SCORE_FIELDS)}


print("=== BASELINE EVALUATION (5 holdout samples) ===\n")
with open('holdout.jsonl') as f:
    holdout_lines = f.readlines()

baseline_scores = []
for i, line in enumerate(holdout_lines[:5]):
    sample = json.loads(line)
    img_bytes = base64.b64decode(sample['image_b64'])
    image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    gt = json.loads(sample['ground_truth'])

    pred = run_inference(image, model, tokenizer)
    result = score_extraction(pred, gt)
    baseline_scores.append(result['completeness'])

    print(f"Sample {i+1}: completeness={result['completeness']:.0%}")
    for field, status in result['field_scores'].items():
        symbol = '✓' if status in ('exact_match', 'correct_null') else '✗'
        print(f"  {symbol} {field}: {status}")
    print()

print(f"Baseline avg completeness: {sum(baseline_scores)/len(baseline_scores):.0%}")
print("\nCell 8 PASSED")

In [ ]:
# Cell 9 — Training
# Unsloth's SFTTrainer handles vision tokenization internally
# when given a dataset that returns {"messages": [...]} dicts.
# No custom collate_fn needed.
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

training_args = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    remove_unused_columns=False,
    report_to="none",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=2048,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
)

total_steps = len(train_dataset) * 3 // 4
print(f"Starting training: {len(train_dataset)} samples, 3 epochs, ~{total_steps} optimizer steps")
print("This should take 20-40 minutes on T4...\n")

trainer_stats = trainer.train()

print(f"\nTraining complete.")
print(f"Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f}min)")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print("\nCell 9 PASSED")

In [ ]:
# Cell 10 — Post-Training Evaluation (all 20 holdout samples)
FastVisionModel.for_inference(model)

print("=== POST-TRAINING EVALUATION (20 holdout samples) ===\n")
post_scores = []

for i, line in enumerate(holdout_lines):
    sample = json.loads(line)
    img_bytes = base64.b64decode(sample['image_b64'])
    image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    gt = json.loads(sample['ground_truth'])

    pred = run_inference(image, model, tokenizer)
    result = score_extraction(pred, gt)
    post_scores.append(result['completeness'])

    print(f"Sample {i+1}: completeness={result['completeness']:.0%}")
    for field, status in result['field_scores'].items():
        symbol = '✓' if status in ('exact_match', 'correct_null') else '✗'
        print(f"  {symbol} {field}: {status}")
    print()

post_avg = sum(post_scores) / len(post_scores)
base_avg = sum(baseline_scores) / len(baseline_scores)
print(f"Post-training avg completeness: {post_avg:.0%}")
print(f"Baseline avg was:               {base_avg:.0%}")
print(f"Delta:                          {post_avg - base_avg:+.0%}")
print("\nCell 10 PASSED")

In [ ]:
# Cell 11 — Export to GGUF
# Unsloth merges LoRA into base weights and exports GGUF + Modelfile in one call.
# This takes 5-15 minutes.

print("Merging adapter and exporting to GGUF (q4_k_m)...")
print("This will take 5-15 minutes. Do not interrupt.\n")

model.save_pretrained_gguf(
    "check-extractor-qwen25vl",
    tokenizer,
    quantization_method="q4_k_m"
)

print("\nGGUF export complete. Files:")
for f in os.listdir("check-extractor-qwen25vl"):
    size_mb = os.path.getsize(f"check-extractor-qwen25vl/{f}") / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

# Print the Modelfile
modelfile_path = None
for f in os.listdir("check-extractor-qwen25vl"):
    if f.lower() == 'modelfile':
        modelfile_path = f"check-extractor-qwen25vl/{f}"
        break

if modelfile_path:
    print("\n=== AUTO-GENERATED MODELFILE ===")
    with open(modelfile_path) as f:
        print(f.read())
else:
    print("\nNo Modelfile generated — you'll need to create one manually (see Cell 13)")

print("\nCell 11 PASSED")

In [ ]:
# Cell 12 — Download the GGUF
import shutil
from google.colab import files

shutil.make_archive("check-extractor-qwen25vl", 'zip', "check-extractor-qwen25vl")

print("Downloading check-extractor-qwen25vl.zip")
print("This is ~4-5GB — it will take a few minutes.")
files.download("check-extractor-qwen25vl.zip")

In [ ]:
# Cell 13 — Instructions for Loading into Ollama
print("""
=== LOADING YOUR FINE-TUNED MODEL INTO OLLAMA ===

After downloading check-extractor-qwen25vl.zip:

1. Unzip it:
   unzip check-extractor-qwen25vl.zip -d check-extractor-qwen25vl

2. IMPORTANT — Upgrade Ollama first (requires >= 0.7.0 for Qwen2.5-VL):
   Your current version is 0.30.10 which does NOT support this model.
   Run: brew upgrade ollama
   Or download latest from: https://ollama.com

3. Create the model in Ollama:
   cd check-extractor-qwen25vl
   ollama create check-extractor -f Modelfile

4. Verify it's registered:
   ollama list

5. Test it:
   ollama run check-extractor "Hello, describe what you can do."

6. Update ollama.py — change the model name:
   From: "model": "minicpm-v"
   To:   "model": "check-extractor"

7. Run batch test:
   python test_ollama.py

NOTES:
- The Modelfile includes the correct Qwen2.5-VL chat template.
  Do not modify the template section.
- If no Modelfile was generated, create one manually:

  FROM ./check-extractor-qwen25vl-unsloth.Q4_K_M.gguf
  PARAMETER temperature 0.1
  SYSTEM You are a bank check data extraction model. Return only valid JSON.

- The model uses Metal acceleration automatically on Apple Silicon.
""")